# Lektion 13 - Agent Hukommelse


## Setup

Denne notesbog demonstrerer, hvordan man bygger en rejsebookingsagent med **vedvarende hukommelse** ved hjælp af **Microsoft Agent Framework** (MAF).

Du vil lære, hvordan forskellige typer agenthukommelse — arbejdshukommelse, korttidshukommelse og langtidshukommelse — påvirker, hvordan en agent fastholder og bruger information på tværs af samtaler.

**Forudsætninger:**
- Et Azure AI Foundry-projekt med en implementeret chatmodel (f.eks. `gpt-4o-mini`).
- Logget ind med Azure CLI — kør `az login` i din terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — dit Azure AI Foundry-projekt-endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — navnet på din implementerede model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Typer af agenthukommelse

AI-agenter kan bruge forskellige typer hukommelse, som hver tjener et bestemt formål:

### Arbejdshukommelse
Selve samtaletråden — de beskeder, der udveksles i en enkelt session. Agenten kan referere tilbage til tidligere beskeder i den samme tråd for at opretholde sammenhæng. I MAF oprettes dette via **`agent.create_session()`**, som returnerer en `AgentSession`.

### Korttidshukommelse
Information, der bevares i løbet af en opgave eller session, men ikke gemmes permanent. For eksempel kan agenten akkumulere fakta under en flertrinsplanlægningssamtale og bruge dem til at lave en endelig rejseplan.

### Langtidshukommelse
Præferencer og fakta, der bevares **på tværs af sessioner**. En tilbagevendende bruger skal ikke gentage deres kostrestriktioner eller rejsepræferencer. Langtidshukommelse understøttes typisk af en ekstern lagerplads — en database, fil eller vektorindeks — og præsenteres for agenten via værktøjer.


## Arbejdshukommelse med sessioner

Den simpleste form for hukommelse er samtalesessionen. Når du sender den samme session (oprettet via `agent.create_session()`) til efterfølgende `agent.run()` kald, kan agenten se hele historikken for den samtale og kan huske tidligere detaljer.

Lad os oprette en rejseagent og demonstrere arbejdshukommelse.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agenten huskede korrekt budgettet, fordi begge beskeder deler den samme session. Dette er **arbejdshukommelse** — den eksisterer kun i sessionens levetid.

### Hvad sker der med en ny tråd?

Hvis vi opretter en **ny** session, har agenten ingen hukommelse om den tidligere samtale:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Langtidshukommelsesmønster

For at huske brugerpræferencer **på tværs af sessioner** har vi brug for en vedvarende lager, der lever uden for samtaletråden. Agenten får adgang til dette lager gennem **værktøjer** — funktioner den kan kalde for at gemme og hente information.

Nedenfor implementerer vi et simpelt præferencelager i hukommelsen (i produktion ville du understøtte dette med en database eller vektorindeks) og eksponerer det som værktøjer, som agenten kan bruge.

### Arkitektur
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — Førstegangsbruger booker en jubilæumsrejse

Sarah besøger for første gang. Agenten skal gemme hendes præferencer via værktøjerne og bruge dem til at anbefale hoteller.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenario 2 — Sarah vender tilbage flere uger senere

Sarah starter en **helt ny tråd** (simulerer en ny session). Arbejdshukommelsen er tom, men den langsigtede præferencebutik har stadig hendes oplysninger. Agenten skal hente dem og bruge dem til at personalisere anbefalinger.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Resumé

I denne lektion lærte du tre typer agenthukommelse og hvordan man implementerer dem med Microsoft Agent Framework:

| Hukommelsestype | MAF-mekanisme | Varighed |
|---|---|---|
| **Arbejdende** | `agent.create_session()` | Enkelt samtale |
| **Kortvarig** | Akkumuleret kontekst inden for en tråd | Enkelt opgave / session |
| **Langvarig** | Ekstern lager tilgået via `@tool` funktioner | På tværs af sessioner |

### Vigtige pointer
1. **`agent.create_session()`** giver arbejdende hukommelse — agenten ser hele samtalehistorikken inden for en session.
2. **Nye sessioner mister kontekst** — uden langvarig hukommelse kan agenten ikke huske tidligere samtaler.
3. **`@tool` funktioner** bygger bro — de lader agenten gemme og hente information fra en vedvarende lager.
4. **Personalisering forbedres over tid** — jo flere præferencer der gemmes, desto bedre bliver agentens anbefalinger.

### Anvendelser i den virkelige verden
- **Kundeservice**: Husk kundehistorik og præferencer
- **Personlige assistenter**: Oprethold kontekst over dage eller uger
- **Sundhedspleje**: Følg patientinformation og præferencer
- **E-handel**: Personliggjort shopping baseret på historik

### Næste skridt
- Erstat den in-memory dict med en database eller vektorlager (f.eks. Azure AI Search)
- Tilføj hukommelsesudløb for tidsfølsomme oplysninger
- Byg multi-agent systemer med delt hukommelse
- Udforsk Cognee-notebooken for vidensgraf-støttet hukommelse


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:  
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, bedes du være opmærksom på, at automatiske oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For vigtig information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
